In [ ]:
from pathlib import Path
import html
import os

import yaml
from IPython.display import HTML, display


def load_qa_config(config_path="../configs/automatic_photometric_qa.yaml"):
    """Load the release QA configuration file."""

    with open(config_path, encoding="utf-8") as config_file:
        config = yaml.safe_load(config_file)

    if not isinstance(config, dict):
        raise ValueError("The release QA configuration must be a YAML mapping.")

    return config


def get_config_section(config, section_name):
    """Return a required configuration section."""

    section = config.get(section_name)

    if not isinstance(section, dict):
        raise ValueError(f"Missing or invalid configuration section: {section_name}")

    return section


config_path = Path(os.environ.get("AUTOMATIC_PHOTOMETRIC_QA_CONFIG", "../configs/automatic_photometric_qa.yaml")).expanduser().resolve()
config_dir = config_path.parent
config = load_qa_config(config_path)

notebook_config = get_config_section(config, "notebook")
cluster_config = get_config_section(config, "cluster")

if "catalogs" in config:
    catalog_configs = config["catalogs"]

    if not isinstance(catalog_configs, list) or not catalog_configs:
        raise ValueError("catalogs must be a non-empty list of catalog configurations.")
else:
    catalog_config = get_config_section(config, "catalog")
    catalog_configs = [
        {
            **catalog_config,
            "title": catalog_config.get("title", notebook_config["title"]),
            "basic_statistics": config.get("basic_statistics"),
            "unique_count": config.get("unique_count"),
            "spatial_distribution": config.get("spatial_distribution"),
            "magnitudes": config.get("magnitudes"),
            "magnitude_errors": config.get("magnitude_errors"),
        }
    ]

for catalog_index, catalog_config in enumerate(catalog_configs, start=1):
    if not isinstance(catalog_config, dict):
        raise ValueError(f"catalogs[{catalog_index}] must be a YAML mapping.")

    if "path" not in catalog_config:
        raise ValueError(f"catalogs[{catalog_index}] is missing required key: path")

    catalog_config.setdefault("title", f"Catalog {catalog_index}")

notebook_title = notebook_config["title"]
notebook_subtitle = notebook_config.get("subtitle", "")
last_verified_run = notebook_config.get("last_verified_run")


def resolve_config_path(path_value):
    """Resolve a configured path relative to the YAML file."""

    candidate = Path(path_value).expanduser()

    if candidate.is_absolute():
        return candidate

    return (config_dir / candidate).resolve()


In [ ]:
logo_html = """
<div style="display: flex; align-items: center; gap: 24px; margin-bottom: 18px;">
  <div style="display: flex; align-items: center; gap: 18px; flex: 0 0 auto;">
    <img src="https://www.linea.org.br/brand/linea-logo-color.svg" width="100">
    <img src="https://cdn2.webdamdb.com/1280_c3PXjCZbPM23.png" width="180">
  </div>
  <div style="min-width: 0;">
    <h1 style="margin: 0 0 8px 0;">{title}</h1>
    {subtitle}
    {last_verified_run}
  </div>
</div>
"""

subtitle_html = ""

if notebook_subtitle:
    subtitle_html = (
        '<p style="font-size: 18px; margin: 0 0 6px 0;">'
        f"{html.escape(notebook_subtitle)}"
        "</p>"
    )

last_verified_run_html = ""

if last_verified_run:
    last_verified_run_html = (
        '<p style="margin: 0;">Last verified run: '
        f"<b>{html.escape(str(last_verified_run))}</b></p>"
    )

display(
    HTML(
        logo_html.format(
            title=html.escape(notebook_title),
            subtitle=subtitle_html,
            last_verified_run=last_verified_run_html,
        )
    )
)


---

This notebook provides lightweight statistics and diagnostic plots for quick characterization of the data product.

---


In [ ]:
import os
import warnings

import dask
import dask.array as da
import dask.dataframe as dd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from dask import delayed
from dask.distributed import Client, LocalCluster
from IPython.display import Markdown
from matplotlib.colors import LogNorm

try:
    from dask_jobqueue import SLURMCluster
except ImportError:
    SLURMCluster = None

warnings.filterwarnings(
    "ignore",
    message="Sending large graph of size.*",
    category=UserWarning,
)
warnings.filterwarnings(
    "ignore",
    message="invalid value encountered in subtract",
    category=RuntimeWarning,
    module=r"dask\.array\.reductions",
)
warnings.filterwarnings(
    "ignore",
    message="invalid value encountered in subtract",
    category=RuntimeWarning,
    module=r"numpy\.lib\._function_base_impl",
)

expected_warning_filters = [
    "ignore:invalid value encountered in subtract:RuntimeWarning:dask.array.reductions",
    "ignore:invalid value encountered in subtract:RuntimeWarning:numpy.lib._function_base_impl",
]
existing_python_warnings = os.environ.get("PYTHONWARNINGS")
os.environ["PYTHONWARNINGS"] = ",".join(
    [
        *([existing_python_warnings] if existing_python_warnings else []),
        *expected_warning_filters,
    ]
)

sns.set_theme(style="whitegrid")


QA_PROGRESS_PREFIX = "QA_PROGRESS:"


def qa_log(message):
    """Emit a progress line for the command-line runner."""

    print(f"{QA_PROGRESS_PREFIX} {message}", flush=True)


def get_basic_statistics_columns(catalog_columns, basic_statistics_config):
    """Resolve the configured basic-statistics column selection."""

    available_columns = list(catalog_columns)
    selection = basic_statistics_config.get("columns")
    default_first_n = int(basic_statistics_config.get("default_first_n", 20))
    max_columns = int(basic_statistics_config.get("max_columns", 100))

    if selection is None:
        return available_columns[:default_first_n]

    if isinstance(selection, str):
        if selection.lower() == "all":
            if not basic_statistics_config.get("allow_all_columns", False):
                raise ValueError(
                    'basic_statistics.columns is set to "all", which can be expensive '
                    "for wide catalogs. Set basic_statistics.allow_all_columns: true "
                    "to confirm that all columns should be processed."
                )

            warnings.warn(
                'basic_statistics.columns="all" was explicitly allowed. All catalog '
                "columns will be processed, which can be expensive for wide catalogs.",
                UserWarning,
            )

            return available_columns

        raise ValueError(
            'basic_statistics.columns must be null, "all", or a list of column names.'
        )

    if isinstance(selection, list):
        missing_columns = [column for column in selection if column not in available_columns]

        if missing_columns:
            raise ValueError(
                "Configured basic-statistics columns are not present in the catalog: "
                + ", ".join(missing_columns)
            )

        if len(selection) > max_columns and not basic_statistics_config.get("allow_many_columns", False):
            raise ValueError(
                f"basic_statistics.columns contains {len(selection)} columns, which exceeds "
                f"basic_statistics.max_columns={max_columns}. This can create large Dask "
                "graphs and heavy reductions. Increase max_columns or set "
                "basic_statistics.allow_many_columns: true to confirm this choice."
            )

        if len(selection) > max_columns:
            warnings.warn(
                f"basic_statistics.columns contains {len(selection)} columns, exceeding "
                f"basic_statistics.max_columns={max_columns}, but allow_many_columns is true. "
                "This run may create large Dask graphs and heavy reductions.",
                UserWarning,
            )

        return selection

    raise ValueError(
        'basic_statistics.columns must be null, "all", or a list of column names.'
    )


def make_dask_cluster(cluster_config):
    """Create a Dask client and cluster from the configured backend."""

    cluster_type = cluster_config.get("type", "local").lower()

    if cluster_type == "local":
        local_config = get_config_section(cluster_config, "local")
        cluster = LocalCluster(
            n_workers=int(local_config.get("n_workers", 3)),
            threads_per_worker=int(local_config.get("cores", 2)),
            memory_limit=local_config.get("memory", "2GB"),
        )
        client = Client(cluster)
        return client, cluster

    if cluster_type == "slurm":
        if SLURMCluster is None:
            raise ImportError(
                "dask_jobqueue is required when cluster.type is set to 'slurm'."
            )

        slurm_config = get_config_section(cluster_config, "slurm")
        cluster = SLURMCluster(
            n_workers=int(slurm_config["n_workers"]),
            queue=slurm_config["queue"],
            account=slurm_config["account"],
            interface=slurm_config["interface"],
            cores=int(slurm_config["cores"]),
            processes=int(slurm_config["processes"]),
            memory=slurm_config["memory"],
            walltime=slurm_config["walltime"],
        )

        adapt_config = slurm_config.get("adapt")

        if adapt_config:
            cluster.adapt(
                minimum_jobs=int(adapt_config["minimum_jobs"]),
                maximum_jobs=int(adapt_config["maximum_jobs"]),
            )

        client = Client(cluster)
        wait_for_workers = slurm_config.get("wait_for_workers")

        if wait_for_workers:
            client.wait_for_workers(int(wait_for_workers))

        return client, cluster

    raise ValueError("cluster.type must be either 'local' or 'slurm'.")


def make_band_model_columns(bands, models):
    """Build band/model column names."""

    return [
        f"{band}_{model}"
        for band in bands
        for model in models
    ]




def model_uses_flux_conversion(model):
    """Return True when a configured model should be converted from flux."""

    return "Flux" in model


def model_uses_flux_error_conversion(model):
    """Return True when a configured error model should be converted from flux error."""

    return "FluxErr" in model


def get_mag_offset(section_config):
    """Return the configured magnitude zero-point offset."""

    mag_offset = section_config.get("mag_offset")

    if mag_offset is None:
        raise ValueError(
            "mag_offset is required when converting Flux columns to magnitudes."
        )

    return float(mag_offset)


def make_flux_magnitude_source(parquet_files, bands, models, section_config):
    """
    Load magnitude inputs and lazily convert Flux columns to magnitudes.

    Non-positive, missing, and non-finite flux values become NaN through the
    lazy Dask expression and are excluded later by the finite-value filters used
    in histograms and distribution statistics.
    """

    output_columns = make_band_model_columns(bands, models)
    conversion_models = [model for model in models if model_uses_flux_conversion(model)]
    mag_offset = get_mag_offset(section_config) if conversion_models else None

    raw_columns = sorted(set(output_columns))
    source = dd.read_parquet(parquet_files, engine="pyarrow", columns=raw_columns)

    for band in bands:
        for model in conversion_models:
            column = f"{band}_{model}"
            finite_positive_flux = source[column].where(
                (source[column] > 0) & da.isfinite(source[column])
            )
            source[column] = mag_offset - 2.5 * np.log10(finite_positive_flux)

    return source[output_columns], output_columns


def flux_error_model_to_flux_model(error_model):
    """Return the matching flux model for a FluxErr model name."""

    if error_model.endswith("FluxErr"):
        return error_model[: -len("Err")]

    raise ValueError(
        "Flux error conversion requires model names ending in 'FluxErr'."
    )


def make_magnitude_error_source(parquet_files, bands, models, section_config):
    """Load error inputs and lazily convert FluxErr columns to magnitude errors."""

    output_columns = make_band_model_columns(bands, models)
    conversion_models = [
        model for model in models if model_uses_flux_error_conversion(model)
    ]

    raw_columns = set(output_columns)

    for model in conversion_models:
        flux_model = flux_error_model_to_flux_model(model)
        raw_columns.update(f"{band}_{flux_model}" for band in bands)

    source = dd.read_parquet(
        parquet_files,
        engine="pyarrow",
        columns=sorted(raw_columns),
    )

    conversion_factor = 2.5 / np.log(10.0)

    for band in bands:
        for model in conversion_models:
            error_column = f"{band}_{model}"
            flux_column = f"{band}_{flux_error_model_to_flux_model(model)}"
            valid = (
                (source[flux_column] > 0)
                & (source[error_column] >= 0)
                & da.isfinite(source[flux_column])
                & da.isfinite(source[error_column])
            )
            converted_error = conversion_factor * source[error_column] / source[flux_column]
            source[error_column] = converted_error.where(valid)

    return source[output_columns], output_columns

def _qa_map_partitions(source, function, *args, meta):
    """Run a partition-level function while suppressing known LSDB warnings."""

    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message="output of the function must be a DataFrame to generate an LSDB.*",
            category=RuntimeWarning,
        )
        return source.map_partitions(function, *args, meta=meta)


# Count and cardinality helpers

def _qa_partition_row_count(partition):
    """Count rows in one partition."""

    return pd.Series([len(partition)], name="count", dtype="int64")


def qa_row_count(source):
    """Compute the total row count for a distributed table."""

    counts = _qa_map_partitions(
        source,
        _qa_partition_row_count,
        meta=pd.Series(name="count", dtype="int64"),
    ).compute()

    return int(counts.sum())


def _qa_partition_value_counts(partition, column):
    """Compute value counts for one partition."""

    return partition[column].value_counts(dropna=False).rename("count")


def qa_value_counts(source, column):
    """Compute exact value counts for a distributed column."""

    partials = _qa_map_partitions(
        source,
        _qa_partition_value_counts,
        column,
        meta=pd.Series(name="count", dtype="int64"),
    ).compute()

    return partials.groupby(level=0, dropna=False).sum().sort_index()


def _qa_partition_unique_values(partition, column, max_unique_values):
    """Return unique values from one partition, capped for driver safety."""

    values = partition[column].drop_duplicates().head(max_unique_values + 1)

    return pd.Series(values.to_numpy(), name=column)


def qa_unique_count(source, column, dropna=True, max_unique_values=10000):
    """
    Compute an exact global unique count with a configured cardinality cap.

    The function either returns an exact global count or raises an error. It does
    not return approximate, sampled, or per-partition counts.
    """

    if max_unique_values <= 0:
        raise ValueError("max_unique_values must be greater than zero.")

    partial_uniques = _qa_map_partitions(
        source,
        _qa_partition_unique_values,
        column,
        int(max_unique_values),
        meta=pd.Series(name=column, dtype=source[column].dtype),
    ).compute()

    if dropna:
        partial_uniques = partial_uniques[~pd.isna(partial_uniques)]

    unique_values = pd.Index(partial_uniques).drop_duplicates()

    if len(unique_values) > max_unique_values:
        raise ValueError(
            f"Exact unique count for column '{column}' exceeded "
            f"unique_count.max_unique_values={max_unique_values}. No approximate "
            "or partial value was reported. Increase max_unique_values only if this "
            "high-cardinality exact count is scientifically required and the driver "
            "has enough memory."
        )

    return int(len(unique_values))


# Histogram helpers

def _qa_partition_histogram2d_array(
    partition,
    ra_column,
    dec_column,
    xedges,
    yedges,
):
    """Compute a fixed-size 2D histogram for one catalog partition."""

    ra = pd.to_numeric(partition[ra_column], errors="coerce").to_numpy()
    dec = pd.to_numeric(partition[dec_column], errors="coerce").to_numpy()

    # Astronomical Mollweide convention: RA = 0 deg at the center,
    # and RA increases to the left.
    x = -np.deg2rad(((ra + 180.0) % 360.0) - 180.0)
    y = np.deg2rad(dec)

    valid = np.isfinite(x) & np.isfinite(y) & (dec >= -90.0) & (dec <= 90.0)
    counts, _, _ = np.histogram2d(x[valid], y[valid], bins=[xedges, yedges])

    return counts.astype("int64", copy=False)


def qa_histogram2d(
    source,
    ra_column,
    dec_column,
    xedges,
    yedges,
    split_every=8,
):
    """Compute an exact distributed 2D histogram."""

    output_shape = (len(xedges) - 1, len(yedges) - 1)
    partition_histograms = []

    for partition in source.to_delayed():
        histogram_delayed = delayed(_qa_partition_histogram2d_array)(
            partition,
            ra_column,
            dec_column,
            xedges,
            yedges,
        )
        histogram_array = da.from_delayed(
            histogram_delayed,
            shape=output_shape,
            dtype="int64",
        )
        partition_histograms.append(histogram_array)

    if not partition_histograms:
        return np.zeros(output_shape, dtype="int64")

    stacked = da.stack(partition_histograms, axis=0)
    total = stacked.sum(axis=0, dtype="int64", split_every=split_every)

    return total.compute()


def _qa_partition_histograms1d_array(partition, columns, edges):
    """Compute fixed-bin 1D histograms for one catalog partition."""

    partition_counts = np.zeros(
        (len(columns), len(edges) - 1),
        dtype="int64",
    )

    for column_index, column in enumerate(columns):
        values = pd.to_numeric(
            partition[column],
            errors="coerce",
        ).to_numpy()

        values = values[np.isfinite(values)]
        counts, _ = np.histogram(values, bins=edges)
        partition_counts[column_index] = counts

    return partition_counts


def qa_histograms1d(
    source,
    columns,
    bins=50,
    value_range=None,
    split_every=8,
):
    """Compute exact histograms for multiple columns in one distributed pass."""

    if value_range is None:
        raise ValueError(
            "value_range must be provided when computing multiple histograms in one pass."
        )

    edges = np.linspace(value_range[0], value_range[1], bins + 1)
    output_shape = (len(columns), bins)
    partition_histograms = []

    for partition in source.to_delayed():
        histogram_delayed = delayed(_qa_partition_histograms1d_array)(
            partition,
            columns,
            edges,
        )
        histogram_array = da.from_delayed(
            histogram_delayed,
            shape=output_shape,
            dtype="int64",
        )
        partition_histograms.append(histogram_array)

    if not partition_histograms:
        empty_counts = {column: np.zeros(bins, dtype="int64") for column in columns}
        return empty_counts, edges

    stacked = da.stack(partition_histograms, axis=0)
    total_counts = stacked.sum(axis=0, dtype="int64", split_every=split_every).compute()

    histograms = {
        column: total_counts[column_index]
        for column_index, column in enumerate(columns)
    }

    return histograms, edges


# Distribution statistics helpers

def _qa_partition_distribution_summary(
    partition,
    columns,
    value_range,
    histogram_edges,
    thresholds,
):
    """
    Compute histogram and scalar summaries for one catalog partition.

    For each column, the output contains histogram counts, the count and sum
    of valid values, counts below configured thresholds, and diagnostic counts
    for non-finite, below-range, and above-range values.
    """

    number_of_bins = len(histogram_edges) - 1
    number_of_thresholds = len(thresholds)
    number_of_summary_fields = 2 + number_of_thresholds + 3

    partition_summary = np.zeros(
        (
            len(columns),
            number_of_bins + number_of_summary_fields,
        ),
        dtype="float64",
    )

    lower_limit, upper_limit = value_range

    for column_index, column in enumerate(columns):
        values = pd.to_numeric(
            partition[column],
            errors="coerce",
        ).to_numpy()

        finite_mask = np.isfinite(values)
        valid_mask = finite_mask & (values >= lower_limit) & (values <= upper_limit)
        valid_values = values[valid_mask]

        histogram_counts, _ = np.histogram(valid_values, bins=histogram_edges)

        field_index = number_of_bins

        # Histogram counts.
        partition_summary[column_index, :number_of_bins] = histogram_counts

        # Valid count.
        partition_summary[column_index, field_index] = valid_values.size
        field_index += 1

        # Sum of valid values.
        partition_summary[column_index, field_index] = valid_values.sum(dtype="float64")
        field_index += 1

        # Counts below configured thresholds.
        for threshold in thresholds:
            partition_summary[column_index, field_index] = np.count_nonzero(
                valid_values <= threshold
            )
            field_index += 1

        # NaN, +inf, and -inf values.
        partition_summary[column_index, field_index] = np.count_nonzero(~finite_mask)
        field_index += 1

        # Finite values below the selected range.
        partition_summary[column_index, field_index] = np.count_nonzero(
            finite_mask & (values < lower_limit)
        )
        field_index += 1

        # Finite values above the selected range.
        partition_summary[column_index, field_index] = np.count_nonzero(
            finite_mask & (values > upper_limit)
        )

    return partition_summary


def qa_distribution_statistics(
    source,
    columns,
    value_range,
    peak_bin_width,
    thresholds=(),
    quantiles=(0.16, 0.50, 0.84, 0.95),
    split_every=8,
):
    """
    Compute distributed descriptive statistics for multiple columns.

    Count, mean, histogram peak, threshold fractions, and diagnostic counts use
    partition-level reductions. Quantiles are calculated directly from filtered
    values through Dask's distributed quantile implementation.
    """

    lower_limit, upper_limit = value_range

    if lower_limit >= upper_limit:
        raise ValueError("value_range must satisfy lower_limit < upper_limit.")

    if peak_bin_width <= 0:
        raise ValueError("peak_bin_width must be greater than zero.")

    thresholds = tuple(thresholds)
    quantiles = tuple(quantiles)

    number_of_bins = int(np.ceil((upper_limit - lower_limit) / peak_bin_width))

    # This guarantees that the first and last edges match value_range.
    histogram_edges = np.linspace(lower_limit, upper_limit, number_of_bins + 1)

    effective_bin_width = histogram_edges[1] - histogram_edges[0]
    histogram_centers = (histogram_edges[:-1] + histogram_edges[1:]) / 2
    number_of_summary_fields = 2 + len(thresholds) + 3
    output_shape = (len(columns), number_of_bins + number_of_summary_fields)

    selected_source = source[columns]
    partition_summaries = []

    for partition in selected_source.to_delayed():
        summary_delayed = delayed(_qa_partition_distribution_summary)(
            partition,
            columns,
            value_range,
            histogram_edges,
            thresholds,
        )

        summary_array = da.from_delayed(
            summary_delayed,
            shape=output_shape,
            dtype="float64",
        )

        partition_summaries.append(summary_array)

    if not partition_summaries:
        return pd.DataFrame(), effective_bin_width

    total_summary = da.stack(partition_summaries, axis=0).sum(
        axis=0,
        dtype="float64",
        split_every=split_every,
    )

    # Values outside the statistical range become NaN.
    # Dask's quantile implementation ignores them.
    filtered_source = selected_source.where(
        (selected_source >= lower_limit) & (selected_source <= upper_limit)
    )

    distributed_quantiles = filtered_source.quantile(q=list(quantiles))

    # Compute reductions and expression-backed quantiles separately to avoid
    # materializing mixed Dask collection types in the driver.
    total_summary_result = total_summary.compute()
    quantile_result = distributed_quantiles.compute()

    histogram_counts = total_summary_result[:, :number_of_bins]
    field_index = number_of_bins

    counts = total_summary_result[:, field_index].astype("int64")
    field_index += 1

    sums = total_summary_result[:, field_index]
    field_index += 1

    threshold_counts = {}

    for threshold in thresholds:
        threshold_counts[threshold] = total_summary_result[:, field_index].astype("int64")
        field_index += 1

    nonfinite_counts = total_summary_result[:, field_index].astype("int64")
    field_index += 1

    below_range_counts = total_summary_result[:, field_index].astype("int64")
    field_index += 1

    above_range_counts = total_summary_result[:, field_index].astype("int64")

    means = np.divide(
        sums,
        counts,
        out=np.full(len(columns), np.nan, dtype="float64"),
        where=counts > 0,
    )

    peak_bin_indices = np.argmax(histogram_counts, axis=1)
    peak_bin_centers = histogram_centers[peak_bin_indices].astype("float64")
    peak_bin_centers[counts == 0] = np.nan

    quantile_names = {
        0.16: "P16",
        0.50: "Median",
        0.84: "P84",
        0.95: "P95",
    }

    statistics_values = {
        "Column": columns,
        "Count": counts,
        "Mean": means,
    }

    for quantile in quantiles:
        column_name = quantile_names.get(quantile, f"P{quantile * 100:g}")
        statistics_values[column_name] = [
            quantile_result.loc[quantile, column]
            for column in columns
        ]

    statistics_values["Peak-bin center"] = peak_bin_centers

    for threshold in thresholds:
        fraction_name = f"Fraction ≤ {threshold:g}"
        statistics_values[fraction_name] = np.divide(
            threshold_counts[threshold],
            counts,
            out=np.full(len(columns), np.nan, dtype="float64"),
            where=counts > 0,
        )

    statistics_values.update(
        {
            "Non-finite": nonfinite_counts,
            "Below range": below_range_counts,
            "Above range": above_range_counts,
        }
    )

    statistics = pd.DataFrame(statistics_values)

    return statistics, effective_bin_width


# Plotting and display helpers

def ra_to_mollweide_x(ra_deg):
    """Convert RA in degrees to Mollweide longitude."""

    ra_centered = ((np.asarray(ra_deg) + 180.0) % 360.0) - 180.0
    return -np.deg2rad(ra_centered)


def plot_wrapped_curve(ax, ra_deg, dec_deg, **plot_kwargs):
    """Plot a curve in Mollweide coordinates, split at RA wrap jumps."""

    x = ra_to_mollweide_x(ra_deg)
    y = np.deg2rad(dec_deg)

    valid = np.isfinite(x) & np.isfinite(y)
    x = x[valid]
    y = y[valid]

    if len(x) < 2:
        return

    jump_idx = np.where(np.abs(np.diff(x)) > np.pi)[0]
    start = 0
    first_segment = True

    for jump in jump_idx:
        end = jump + 1

        if end - start > 1:
            kwargs = plot_kwargs

            if not first_segment:
                kwargs = {
                    key: value
                    for key, value in plot_kwargs.items()
                    if key != "label"
                }

            ax.plot(x[start:end], y[start:end], **kwargs)
            first_segment = False

        start = end

    if len(x) - start > 1:
        kwargs = plot_kwargs

        if not first_segment:
            kwargs = {
                key: value
                for key, value in plot_kwargs.items()
                if key != "label"
            }

        ax.plot(x[start:], y[start:], **kwargs)


def add_band_model_metadata(
    statistics,
    bands,
    models,
    model_labels,
):
    """Add readable band and model columns to a QA table."""

    column_metadata = pd.DataFrame(
        [
            {
                "Column": f"{band}_{model}",
                "Band": band,
                "Model": model_labels.get(model, model),
            }
            for band in bands
            for model in models
        ]
    )

    return column_metadata.merge(
        statistics,
        on="Column",
        how="left",
    ).drop(columns="Column")


def style_distribution_statistics(
    statistics,
    caption,
):
    """Apply consistent formatting to a distribution statistics table."""

    formatters = {
        "Count": "{:,}",
        "Mean": "{:.4f}",
        "Median": "{:.4f}",
        "P16": "{:.4f}",
        "P84": "{:.4f}",
        "P95": "{:.4f}",
        "Peak-bin center": "{:.4f}",
        "Non-finite": "{:,}",
        "Below range": "{:,}",
        "Above range": "{:,}",
    }

    for column in statistics.columns:
        if column.startswith("Fraction ≤"):
            formatters[column] = "{:.2%}"

    # Use only formatters corresponding to columns actually present.
    formatters = {
        column: formatter
        for column, formatter in formatters.items()
        if column in statistics.columns
    }

    return (
        statistics.style
        .format(formatters)
        .hide(axis="index")
        .set_caption(caption)
    )


## Distributed execution


Create the configured Dask cluster for lazy QA operations.


In [ ]:
client, cluster = make_dask_cluster(cluster_config)

print(client)
print(cluster)


In [ ]:
def resolve_catalog_files(catalog_config):
    """Resolve one catalog path and return matching parquet files."""

    path_to_catalog = resolve_config_path(catalog_config["path"])
    parquet_pattern = catalog_config.get("parquet_pattern", "*.parquet")

    if path_to_catalog.is_file():
        parquet_files = [path_to_catalog]
    else:
        parquet_files = sorted(path_to_catalog.rglob(parquet_pattern))

    if not parquet_files:
        raise FileNotFoundError(
            f"No parquet files found for catalog path: {path_to_catalog}"
        )

    return path_to_catalog, parquet_files


def format_bytes(size_bytes):
    """Format a byte count with binary units."""

    units = ("B", "KiB", "MiB", "GiB", "TiB", "PiB")
    value = float(size_bytes)

    for unit in units:
        if value < 1024.0 or unit == units[-1]:
            return f"{value:.1f} {unit}" if unit != "B" else f"{int(value)} {unit}"
        value /= 1024.0


def catalog_size_bytes(path_to_catalog, parquet_files):
    """Compute catalog size from selected parquet files."""

    if path_to_catalog.is_file():
        return path_to_catalog.stat().st_size

    return sum(path.stat().st_size for path in parquet_files)


def display_section_heading(title):
    """Display a third-level report section heading."""

    display(Markdown(f"### {title}"))


def display_catalog_columns(columns):
    """Display catalog column names in a scrollable text area."""

    cols_text = "\n".join(map(str, columns))
    display(
        HTML(
            f"""
            <textarea
                rows="10"
                style="width: 100%; font-family: monospace; white-space: pre;"
                readonly
            >{html.escape(cols_text)}</textarea>
            """
        )
    )


def render_catalog_basics(catalog_config):
    """Render required size, row-count, and column-count information."""

    display_section_heading("Basic Product Information")
    qa_log(f"[{catalog_config['title']}] Resolving catalog input files")
    path_to_catalog, parquet_files = resolve_catalog_files(catalog_config)
    catalog = dd.read_parquet(parquet_files, engine="pyarrow")
    qa_log(f"[{catalog_config['title']}] Computing catalog size")

    print("Catalog path:", path_to_catalog)
    print("Parquet files:", f"{len(parquet_files):,}")
    print("Catalog size:", format_bytes(catalog_size_bytes(path_to_catalog, parquet_files)))

    qa_log(f"[{catalog_config['title']}] Computing total row count")
    count_column = catalog.columns[0]
    count_data = dd.read_parquet(parquet_files, engine="pyarrow", columns=[count_column])
    qa_total_rows = qa_row_count(count_data)
    del count_data

    print("Total rows:", f"{qa_total_rows:,}")
    qa_log(f"[{catalog_config['title']}] Computing number of columns")
    print("Total columns:", f"{len(catalog.columns):,}")
    qa_log(f"[{catalog_config['title']}] Rendering column names")
    display_catalog_columns(catalog.columns)

    return catalog, parquet_files, path_to_catalog


def render_unique_count(catalog_config, parquet_files):
    """Render exact unique-count diagnostics when configured."""

    unique_count_config = catalog_config.get("unique_count")

    if unique_count_config is None:
        return

    qa_log(f"[{catalog_config['title']}] Computing unique count")
    display_section_heading("Unique Count")
    unique_column = unique_count_config["column"]
    max_unique_values = int(unique_count_config.get("max_unique_values", 10000))

    print(f"Computing exact global unique count for column '{unique_column}'.")
    print(f"Driver collection cap: {max_unique_values:,} unique values.")

    unique_data = dd.read_parquet(parquet_files, engine="pyarrow", columns=[unique_column])
    qa_unique_values = qa_unique_count(
        unique_data,
        unique_column,
        max_unique_values=max_unique_values,
    )

    print(f"Exact global unique count for column '{unique_column}': {qa_unique_values:,}")
    del unique_data


def render_basic_statistics(catalog_config, catalog, parquet_files):
    """Render basic descriptive statistics when configured."""

    basic_statistics_config = catalog_config.get("basic_statistics")

    if basic_statistics_config is None:
        return

    qa_log(f"[{catalog_config['title']}] Computing basic statistics")
    display_section_heading("Basic Statistics for Selected Columns")
    statistics_columns = get_basic_statistics_columns(
        catalog.columns,
        basic_statistics_config,
    )

    statistics_data = dd.read_parquet(
        parquet_files,
        engine="pyarrow",
        columns=statistics_columns,
    )

    statistics = statistics_data.describe().compute()
    statistics_html = (
        '<div style="max-height: 520px; overflow: auto;">'
        + statistics.to_html(max_rows=None, max_cols=None)
        + "</div>"
    )

    display(HTML(statistics_html))
    del statistics_data


def render_spatial_distribution(catalog_config, parquet_files):
    """Render the spatial distribution plot when configured."""

    spatial_distribution_config = catalog_config.get("spatial_distribution")

    if spatial_distribution_config is None:
        return

    qa_log(f"[{catalog_config['title']}] Generating spatial distribution plot")
    display_section_heading("Spatial Distribution")
    plot_data = dd.read_parquet(
        parquet_files,
        engine="pyarrow",
        columns=[
            spatial_distribution_config["ra_column"],
            spatial_distribution_config["dec_column"],
        ],
    )

    xbins = np.linspace(-np.pi, np.pi, int(spatial_distribution_config["ra_edge_count"]))
    ybins = np.linspace(
        -np.pi / 2.0,
        np.pi / 2.0,
        int(spatial_distribution_config["dec_edge_count"]),
    )

    H = qa_histogram2d(
        plot_data,
        spatial_distribution_config["ra_column"],
        spatial_distribution_config["dec_column"],
        xbins,
        ybins,
        split_every=int(spatial_distribution_config.get("split_every", 8)),
    )

    H = np.ma.masked_equal(H, 0)
    fig = plt.figure(figsize=(16, 8))
    ax = fig.add_subplot(111, projection="mollweide")

    if H.count() > 0:
        mesh = ax.pcolormesh(
            xbins,
            ybins,
            H.T,
            norm=LogNorm(),
            shading="auto",
            cmap="viridis",
            zorder=1,
        )
        cbar = fig.colorbar(mesh, ax=ax, pad=0.05)
        cbar.set_label("Number of objects")
    else:
        ax.text(
            0.5,
            0.5,
            "No finite coordinate pairs",
            transform=ax.transAxes,
            ha="center",
            va="center",
        )

    ax.grid(False)

    dec_grid = np.deg2rad(np.linspace(-90, 90, 500))
    for grid_ra_deg in np.arange(-150, 181, 30):
        ax.plot(
            np.full_like(dec_grid, np.deg2rad(grid_ra_deg)),
            dec_grid,
            color="gray",
            linewidth=0.6,
            alpha=0.5,
            zorder=2,
        )

    ra_grid = np.deg2rad(np.linspace(-180, 180, 800))
    for grid_dec_deg in np.arange(-75, 76, 15):
        ax.plot(
            ra_grid,
            np.full_like(ra_grid, np.deg2rad(grid_dec_deg)),
            color="gray",
            linewidth=0.6,
            alpha=0.5,
            zorder=2,
        )

    for footprint_config in spatial_distribution_config.get("footprints", []):
        footprint = pd.read_csv(resolve_config_path(footprint_config["path"]))
        footprint_type = footprint_config.get("type", "curve")

        if footprint_type == "regions":
            first_curve = True
            region_column = footprint_config.get("region_column", "region_id")
            ring_type_column = footprint_config.get("ring_type_column", "ring_type")
            exterior_value = footprint_config.get("exterior_value", "exterior")
            vertex_column = footprint_config.get("vertex_column", "vertex_id")

            for _, footprint_region in footprint.groupby(region_column, sort=False):
                exterior = (
                    footprint_region[footprint_region[ring_type_column] == exterior_value]
                    .sort_values(vertex_column)
                )

                plot_wrapped_curve(
                    ax,
                    exterior[footprint_config["ra_column"]].to_numpy(),
                    exterior[footprint_config["dec_column"]].to_numpy(),
                    linewidth=footprint_config.get("linewidth", 1),
                    color=footprint_config.get("color"),
                    zorder=footprint_config.get("zorder", 3),
                    label=footprint_config["label"] if first_curve else "_nolegend_",
                )
                first_curve = False

        elif footprint_type == "curve":
            sort_by = footprint_config.get("sort_by")
            if sort_by:
                footprint = footprint.sort_values(sort_by)

            plot_wrapped_curve(
                ax,
                footprint[footprint_config["ra_column"]].to_numpy(),
                footprint[footprint_config["dec_column"]].to_numpy(),
                linewidth=footprint_config.get("linewidth", 1),
                color=footprint_config.get("color"),
                zorder=footprint_config.get("zorder", 3),
                label=footprint_config["label"],
            )
        else:
            raise ValueError(f"Unsupported footprint type: {footprint_type}")

    tick_degs = np.array([-150, -120, -90, -60, -30, 0, 30, 60, 90, 120, 150])
    tick_labels = [
        "150 deg",
        "120 deg",
        "90 deg",
        "60 deg",
        "30 deg",
        "0 deg",
        "330 deg",
        "300 deg",
        "270 deg",
        "240 deg",
        "210 deg",
    ]

    ax.set_xticks(np.deg2rad(tick_degs))
    ax.set_xticklabels(tick_labels)
    ax.set_xlabel(spatial_distribution_config["ra_column"])
    ax.set_ylabel(spatial_distribution_config["dec_column"])
    ax.set_title(
        f"{catalog_config['title']} - {spatial_distribution_config['title_suffix']}"
    )

    handles, labels = ax.get_legend_handles_labels()
    if handles:
        ax.legend(loc="upper right")

    plt.tight_layout()
    plt.show()
    del plot_data
    del H


def render_magnitude_histograms(catalog_config, parquet_files):
    """Render magnitude histograms when configured."""

    magnitudes_config = catalog_config.get("magnitudes")

    if magnitudes_config is None:
        return

    qa_log(f"[{catalog_config['title']}] Generating magnitude histograms")
    display_section_heading("Magnitude Histogram")
    bands = magnitudes_config["bands"]
    magnitude_models = magnitudes_config["models"]
    model_labels = magnitudes_config["model_labels"]
    model_colors = magnitudes_config["model_colors"]
    magnitude_histogram_config = magnitudes_config["histogram"]

    plot_data, magnitude_columns = make_flux_magnitude_source(
        parquet_files,
        bands=bands,
        models=magnitude_models,
        section_config=magnitudes_config,
    )

    magnitude_histograms, magnitude_edges = qa_histograms1d(
        plot_data,
        columns=magnitude_columns,
        bins=int(magnitude_histogram_config["bins"]),
        value_range=tuple(magnitude_histogram_config["range"]),
        split_every=int(magnitude_histogram_config.get("split_every", 8)),
    )

    fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(14, 15), sharex=True)
    axes = axes.ravel()

    for ax, band in zip(axes, bands):
        for model in magnitude_models:
            column = f"{band}_{model}"
            ax.stairs(
                values=magnitude_histograms[column],
                edges=magnitude_edges,
                label=model_labels.get(model, model),
                color=model_colors.get(model),
                linewidth=1.8,
            )

        ax.set_title(f"{band}-band magnitude")
        ax.set_xlabel("Magnitude")
        ax.set_ylabel("Count")
        ax.set_xlim(magnitude_histogram_config["range"][0], magnitude_histogram_config["range"][1])
        ax.set_yscale("log")
        ax.grid(alpha=0.2)
        ax.legend(title="Model")
        ax.tick_params(axis="x", which="both", labelbottom=True)

    fig.suptitle("Magnitude Distributions by Model", fontsize=16, y=1.01)
    plt.tight_layout()
    plt.show()
    del plot_data
    del magnitude_histograms
    del magnitude_edges


def render_magnitude_statistics(catalog_config, parquet_files):
    """Render magnitude statistics when configured."""

    magnitudes_config = catalog_config.get("magnitudes")

    if magnitudes_config is None:
        return

    qa_log(f"[{catalog_config['title']}] Computing magnitude statistics")
    display_section_heading("Magnitude Statistics")
    bands = magnitudes_config["bands"]
    magnitude_models = magnitudes_config["models"]
    model_labels = magnitudes_config["model_labels"]
    magnitude_statistics_config = magnitudes_config["statistics"]

    statistics_data, magnitude_columns = make_flux_magnitude_source(
        parquet_files,
        bands=bands,
        models=magnitude_models,
        section_config=magnitudes_config,
    )

    statistics_range = tuple(magnitude_statistics_config["range"])
    peak_bin_width = float(magnitude_statistics_config["peak_bin_width"])
    magnitude_statistics, effective_bin_width = qa_distribution_statistics(
        statistics_data,
        columns=magnitude_columns,
        value_range=statistics_range,
        peak_bin_width=peak_bin_width,
        thresholds=tuple(magnitude_statistics_config.get("thresholds", [])),
        quantiles=tuple(magnitude_statistics_config.get("quantiles", [0.16, 0.50, 0.84, 0.95])),
        split_every=int(magnitude_statistics_config.get("split_every", 8)),
    )

    magnitude_statistics = add_band_model_metadata(
        magnitude_statistics,
        bands=bands,
        models=magnitude_models,
        model_labels=model_labels,
    )

    print(
        "Statistics include only finite magnitude values in the range "
        f"{statistics_range[0]} ≤ magnitude ≤ {statistics_range[1]}."
    )

    if any(model_uses_flux_conversion(model) for model in magnitude_models):
        print(
            "Flux columns were converted to magnitudes with "
            f"magnitude = {float(magnitudes_config['mag_offset']):g} - 2.5 log10(flux). "
            "Non-positive, missing, and non-finite flux values are converted to NaN "
            "and excluded from finite-value histograms and statistics."
        )

    print(
        "Mean, median, P16, P84, and P95 are calculated directly from the filtered values. "
        f"Peak-bin center is the center of the most populated {effective_bin_width:.3g}-mag bin "
        "and should be interpreted as an empirical distribution turnover, not as a calibrated completeness limit."
    )
    print(
        "Non-finite, below-range, and above-range values are reported separately and excluded from the descriptive statistics."
    )

    display(
        style_distribution_statistics(
            magnitude_statistics,
            caption="Magnitude statistics by band and measurement model",
        )
    )
    del statistics_data
    del magnitude_statistics


def render_magnitude_error_histograms(catalog_config, parquet_files):
    """Render magnitude-error histograms when configured."""

    magnitude_errors_config = catalog_config.get("magnitude_errors")

    if magnitude_errors_config is None:
        return

    qa_log(f"[{catalog_config['title']}] Generating magnitude-error histograms")
    display_section_heading("Magnitude Error Histogram")
    magnitudes_config = catalog_config.get("magnitudes")
    bands = magnitude_errors_config.get(
        "bands",
        magnitudes_config["bands"] if magnitudes_config else None,
    )

    if bands is None:
        raise ValueError(
            "magnitude_errors.bands is required when magnitudes is not configured."
        )

    magnitude_error_models = magnitude_errors_config["models"]
    model_labels = magnitude_errors_config["model_labels"]
    model_colors = magnitude_errors_config["model_colors"]
    magnitude_error_histogram_config = magnitude_errors_config["histogram"]

    plot_data, magnitude_error_columns = make_magnitude_error_source(
        parquet_files,
        bands=bands,
        models=magnitude_error_models,
        section_config=magnitude_errors_config,
    )

    magnitude_error_histograms, magnitude_error_edges = qa_histograms1d(
        plot_data,
        columns=magnitude_error_columns,
        bins=int(magnitude_error_histogram_config["bins"]),
        value_range=tuple(magnitude_error_histogram_config["range"]),
        split_every=int(magnitude_error_histogram_config.get("split_every", 8)),
    )

    fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(14, 15), sharex=True)
    axes = axes.ravel()

    for ax, band in zip(axes, bands):
        for model in magnitude_error_models:
            column = f"{band}_{model}"
            ax.stairs(
                values=magnitude_error_histograms[column],
                edges=magnitude_error_edges,
                label=model_labels.get(model, model),
                color=model_colors.get(model),
                linewidth=1.8,
            )

        ax.set_title(f"{band}-band magnitude error")
        ax.set_xlabel("Magnitude error")
        ax.set_ylabel("Count")
        ax.set_xlim(
            magnitude_error_histogram_config["range"][0],
            magnitude_error_histogram_config["range"][1],
        )
        ax.set_yscale("log")
        ax.grid(alpha=0.2)
        ax.legend(title="Model")
        ax.tick_params(axis="x", which="both", labelbottom=True)

    fig.suptitle("Magnitude Error Distributions by Model", fontsize=16, y=1.01)
    plt.tight_layout()
    plt.show()
    del plot_data
    del magnitude_error_histograms
    del magnitude_error_edges


def render_magnitude_error_statistics(catalog_config, parquet_files):
    """Render magnitude-error statistics when configured."""

    magnitude_errors_config = catalog_config.get("magnitude_errors")

    if magnitude_errors_config is None:
        return

    qa_log(f"[{catalog_config['title']}] Computing magnitude-error statistics")
    display_section_heading("Magnitude Error Statistics")
    magnitudes_config = catalog_config.get("magnitudes")
    bands = magnitude_errors_config.get(
        "bands",
        magnitudes_config["bands"] if magnitudes_config else None,
    )

    if bands is None:
        raise ValueError(
            "magnitude_errors.bands is required when magnitudes is not configured."
        )

    magnitude_error_models = magnitude_errors_config["models"]
    model_labels = magnitude_errors_config["model_labels"]
    magnitude_error_statistics_config = magnitude_errors_config["statistics"]

    statistics_data, magnitude_error_columns = make_magnitude_error_source(
        parquet_files,
        bands=bands,
        models=magnitude_error_models,
        section_config=magnitude_errors_config,
    )

    statistics_range = tuple(magnitude_error_statistics_config["range"])
    peak_bin_width = float(magnitude_error_statistics_config["peak_bin_width"])
    error_thresholds = tuple(magnitude_error_statistics_config.get("thresholds", []))

    magnitude_error_statistics, effective_bin_width = qa_distribution_statistics(
        statistics_data,
        columns=magnitude_error_columns,
        value_range=statistics_range,
        peak_bin_width=peak_bin_width,
        thresholds=error_thresholds,
        quantiles=tuple(magnitude_error_statistics_config.get("quantiles", [0.16, 0.50, 0.84, 0.95])),
        split_every=int(magnitude_error_statistics_config.get("split_every", 8)),
    )

    magnitude_error_statistics = add_band_model_metadata(
        magnitude_error_statistics,
        bands=bands,
        models=magnitude_error_models,
        model_labels=model_labels,
    )

    print(
        "Statistics include only finite magnitude-error values in the range "
        f"{statistics_range[0]} ≤ magnitude error ≤ {statistics_range[1]}."
    )

    if any(model_uses_flux_error_conversion(model) for model in magnitude_error_models):
        print(
            "Flux-error columns were converted to magnitude errors with "
            "sigma_mag = 2.5 / ln(10) * flux_err / flux. Non-positive, missing, "
            "and non-finite flux or flux-error values are converted to NaN and "
            "excluded from finite-value histograms and statistics."
        )

    print(
        "Mean, median, P16, P84, and P95 are calculated directly from the filtered values. "
        f"Peak-bin center is the center of the most populated {effective_bin_width:.3g}-mag bin."
    )
    print(
        "Threshold fractions use Count as their denominator. Non-finite, below-range, "
        "and above-range values are reported separately and excluded from the descriptive statistics."
    )

    display(
        style_distribution_statistics(
            magnitude_error_statistics,
            caption="Magnitude-error statistics by band and measurement model",
        )
    )
    del statistics_data
    del magnitude_error_statistics


def render_catalog_report(catalog_config, catalog_index):
    """Render all configured QA sections for one catalog."""

    catalog_title = catalog_config.get("title", f"Catalog {catalog_index}")
    qa_log(f"Starting catalog {catalog_index}: {catalog_title}")
    print(f"Processing catalog {catalog_index}: {catalog_title}")
    display(Markdown(f"## {catalog_title}"))

    catalog, parquet_files, _ = render_catalog_basics(catalog_config)
    render_unique_count(catalog_config, parquet_files)
    render_basic_statistics(catalog_config, catalog, parquet_files)
    render_spatial_distribution(catalog_config, parquet_files)
    render_magnitude_histograms(catalog_config, parquet_files)
    render_magnitude_statistics(catalog_config, parquet_files)
    render_magnitude_error_histograms(catalog_config, parquet_files)
    render_magnitude_error_statistics(catalog_config, parquet_files)
    del catalog


for catalog_index, catalog_config in enumerate(catalog_configs, start=1):
    render_catalog_report(catalog_config, catalog_index)


In [ ]:
client.close()
cluster.close()
